In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("เชื่อมต่อ Drive สำเร็จ!")

Mounted at /content/drive
เชื่อมต่อ Drive สำเร็จ!


In [2]:
!pip install -q pycocotools mediapipe opencv-python-headless tqdm
import cv2, mediapipe as mp, numpy as np
print("ติดตั้งเครื่องมือ MediaPipe และ OpenCV เรียบร้อย!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.8 MB/s eta 0:00:00
ติดตั้งเครื่องมือ MediaPipe และ OpenCV เรียบร้อย!


In [3]:
import os

# กำหนด Path หลักที่เคยสร้างไว้ใน Notebook แรก
BASE = '/content/drive/MyDrive/exercise_pose'
ANNO = f'{BASE}/data/coco/annotations'
IMGS = f'{BASE}/data/coco/images'
PROC = f'{BASE}/data/processed'

print(f"ระบบพร้อมทำงานที่โฟลเดอร์: {BASE}")

ระบบพร้อมทำงานที่โฟลเดอร์: /content/drive/MyDrive/exercise_pose


In [4]:
import pickle
import numpy as np
import os

# ใช้ Path ที่เรา Copy มาเมื่อวาน (ตรวจสอบให้แน่ใจว่ารัน Setup Cell ก่อนนะครับ)
FILE_PATH = f'{PROC}/samples_val_filtered.pkl'

with open(FILE_PATH, 'rb') as f:
    samples = pickle.load(f)

print(f"✅ โหลดข้อมูลสำเร็จ! พบข้อมูลคุณภาพสูง: {len(samples):,} รายการ")

✅ โหลดข้อมูลสำเร็จ! พบข้อมูลคุณภาพสูง: 60,233 รายการ


In [5]:
def calc_angle(a, b, c):
    """
    คำนวณมุมที่จุด b (หน่วย: องศา) จากพิกัด 3 จุด
    a, b, c: พิกัด [x, y] ของแต่ละจุด
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    # สร้างเวกเตอร์จากจุด b ไปยัง a และ c
    ba = a - b
    bc = c - b

    # คำนวณหา Cosine ของมุมระหว่างเวกเตอร์ด้วย Dot Product
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-7)
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))

    return np.degrees(angle)

# ทดลองคำนวณมุม 90 องศา (สมมติจุด)
print(f"ทดสอบมุมฉาก: {calc_angle([0, 1], [0, 0], [1, 0]):.1f} องศา")

ทดสอบมุมฉาก: 90.0 องศา


In [6]:
# ดึงข้อมูลคนแรก
person = samples[0]
kps = person['kps'] # พิกัด (17, 3) -> [x, y, v]

# ดึงเฉพาะค่า X, Y ของ ขาข้างซ้าย
left_hip = kps[11][:2]
left_knee = kps[13][:2]
left_ankle = kps[15][:2]

# คำนวณมุมเข่าซ้าย
knee_angle = calc_angle(left_hip, left_knee, left_ankle)

print(f"คนแรกในข้อมูล: Image ID {person['img_id']}")
print(f"มุมเข่าซ้ายปัจจุบัน: {knee_angle:.2f} องศา")

คนแรกในข้อมูล: Image ID 537548
มุมเข่าซ้ายปัจจุบัน: 90.00 องศา


In [7]:
def extract_features(kps):
    # kps: np.array shape (17, 3) = [x, y, visibility]
    # return: dict vao features

    f = {}

    # มุมเข่าซ้าย: hip(11) - knee(13) - ankle(15)
    if all(kps[i, 2] > 0 for i in [11, 13, 15]):
        f['left_knee'] = calc_angle(
            kps[11,:2], kps[13,:2], kps[15,:2])

    # มุมเข่าขวา: hip(12) - knee(14) - ankle(16)
    if all(kps[i, 2] > 0 for i in [12, 14, 16]):
        f['right_knee'] = calc_angle(
            kps[12,:2], kps[14,:2], kps[16,:2])

    # มุมสะโพกซ้าย: shoulder(5) - hip(11) - knee(13)
    if all(kps[i, 2] > 0 for i in [5, 11, 13]):
        f['left_hip'] = calc_angle(
            kps[5,:2], kps[11,:2], kps[13,:2])

    # มุมสะโพกขวา: shoulder(6) - hip(12) - knee(14)
    if all(kps[i, 2] > 0 for i in [6, 12, 14]):
        f['right_hip'] = calc_angle(
            kps[6,:2], kps[12,:2], kps[14,:2])

    # ความสมมาตร ซ้าย-ขวา
    if 'left_knee' in f and 'right_knee' in f:
        f['knee_symmetry'] = abs(f['left_knee'] - f['right_knee'])

    return f

In [8]:
# 1. ดึงพิกัด (kps) ของคนลำดับแรกสุดใน Dataset มาเป็นหนูทดลอง
person_kps = samples[0]['kps']

# 2. โยนพิกัดเข้าไปใน "เครื่องปั่น" (ฟังก์ชัน) ที่เราเพิ่งสร้าง
features_result = extract_features(person_kps)

# 3. พิมพ์ผลลัพธ์ที่ได้ออกมาดู
print("🎯 สกัดฟีเจอร์สำเร็จ! นี่คือองศาข้อต่อที่เครื่องคำนวณได้:")
for joint_name, angle_value in features_result.items():
    print(f"- {joint_name}: {angle_value:.1f} องศา")

🎯 สกัดฟีเจอร์สำเร็จ! นี่คือองศาข้อต่อที่เครื่องคำนวณได้:
- right_knee: 67.2 องศา
- right_hip: 86.0 องศา


In [9]:
import pandas as pd
from tqdm.notebook import tqdm

rows = []
for s in tqdm(samples, desc="Extracting features"):
    feat = extract_features(s['kps'])
    if feat:
        feat['img_id']= s['img_id']
        feat['area'] = s['area']
        rows.append(feat)

df = pd.DataFrame(rows)
print(df.shape)
print(df.describe().round(1))

# เซฟเป็น CSV และ pkl
df.to_csv(f'{PROC}/features_val.csv', index=False)
df.to_pickle(f'{PROC}/features_val.pkl')
print(f"เซฟแล้ว: {len(df):,} rows, {df.shape[1]} features")

Extracting features:   0%|          | 0/60233 [00:00<?, ?it/s]

(60042, 7)
       right_knee  right_hip    img_id      area  left_knee  left_hip  \
count     49286.0    57089.0   60042.0   60042.0    49244.0   57041.0   
mean        145.8      147.2  290836.0   17852.4      146.2     146.9   
std          41.1       38.4  168621.3   18263.2       41.0      38.7   
min           0.1        0.0      77.0    3000.3        0.2       0.0   
25%         129.1      132.8  143158.0    5898.8      130.3     132.6   
50%         164.9      164.7  293785.0   11376.3      165.3     164.5   
75%         175.5      173.9  436696.0   22870.1      175.5     173.7   
max         180.0      180.0  581921.0  245513.3      180.0     180.0   

       knee_symmetry  
count        46047.0  
mean            17.4  
std             22.2  
min              0.0  
25%              3.2  
50%              8.7  
75%             22.9  
max            177.6  
เซฟแล้ว: 60,042 rows, 7 features


In [20]:
import os
# ตั้งค่า git (ครั้งแรกครั้งเดียว)
!git config -- global user.email "wanasnan.wangmad@gmail.com"
!git config -- global user.name "Wanasnan Wangmad"

# clone repo เข้า Colab
REPO = "https://github.com/wanasnanwangmad-cmd/exercise-pose.git"
!git clone {REPO} /content/exercise-pose
print("Clone สำเร็จ!")

error: key does not contain a section: global
error: key does not contain a section: global
fatal: destination path '/content/exercise-pose' already exists and is not an empty directory.
Clone สำเร็จ!


In [26]:
import os

BASE = '/content/drive/MyDrive/exercise_pose'

# เช็คทุก folder ใน exercise_pose
for root, dirs, files in os.walk(BASE):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))

In [27]:
import shutil, os

# Colab จะเซฟไว้ที่นี่ตาม default
NOTEBOOK_DIR = '/content/drive/MyDrive/Colab Notebooks'

for nb in os.listdir(NOTEBOOK_DIR):
    if nb.endswith('.ipynb'):
        print(nb)

Untitled0.ipynb
Untitled1.ipynb
344-341 Example-ReadAndWriteFile.ipynb
Untitled2.ipynb
Untitled3.ipynb
Untitled4.ipynb
Untitled6.ipynb
11111.ipynb
Untitled5.ipynb
01_setup_and_data.ipynb
02_explore.ipynb
03_pipeline.ipynb
04_features.ipynb
สำเนาของ 04_features.ipynb
สำเนาของ 02_explore.ipynb
สำเนาของ 03_pipeline.ipynb


In [28]:
import shutil, os

# เปลี่ยน path ตามที่เจอจริงๆ
NOTEBOOK_DIR = '/content/drive/MyDrive/Colab Notebooks'  # หรือ path ที่เจอ

for nb in ['01_setup_and_data.ipynb','02_explore.ipynb',
           '03_pipeline.ipynb','04_features.ipynb']:
    src = f"{NOTEBOOK_DIR}/{nb}"
    dst = f"/content/exercise-pose/{nb}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ copied: {nb}")
    else:
        print(f"❌ not found: {nb}")

✅ copied: 01_setup_and_data.ipynb
✅ copied: 02_explore.ipynb
✅ copied: 03_pipeline.ipynb
✅ copied: 04_features.ipynb


In [29]:
import os, shutil

# ── 1. ตั้งค่า git (ทำครั้งแรกครั้งเดียว) ─────────────────────────
!git config --global user.email "wanasnan.wangmad@gmail.com"
!git config --global user.name "Wanasnan Wangmad"

# ── 2. clone repo จาก GitHub ───────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/exercise-pose.git /content/exercise-pose

# ── 3. copy notebooks จาก Drive เข้า repo ─────────────────────────
NOTEBOOK_DIR = '/content/drive/MyDrive/Colab Notebooks'  # path ที่เจอ
REPO_DIR     = '/content/exercise-pose'

for nb in os.listdir(NOTEBOOK_DIR):
    if nb.endswith('.ipynb'):
        src = f"{NOTEBOOK_DIR}/{nb}"
        dst = f"{REPO_DIR}/{nb}"
        shutil.copy2(src, dst)
        print(f"copied: {nb}")

# ── 4. สร้าง README ถ้ายังไม่มี ────────────────────────────────────
readme = f"{REPO_DIR}/README.md"
if not os.path.exists(readme):
    with open(readme, 'w') as f:
        f.write("# Exercise Pose AI\nPhase 1 — COCO dataset pipeline\n")
    print("created README.md")

# ── 5. commit และ push ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!git add .
!git status
!git commit -m "add notebooks: setup, explore, pipeline, features"
!git push origin main

fatal: destination path '/content/exercise-pose' already exists and is not an empty directory.
copied: Untitled0.ipynb
copied: Untitled1.ipynb
copied: 344-341 Example-ReadAndWriteFile.ipynb
copied: Untitled2.ipynb
copied: Untitled3.ipynb
copied: Untitled4.ipynb
copied: Untitled6.ipynb
copied: 11111.ipynb
copied: Untitled5.ipynb
copied: 01_setup_and_data.ipynb
copied: 02_explore.ipynb
copied: 03_pipeline.ipynb
copied: 04_features.ipynb
copied: สำเนาของ 04_features.ipynb
copied: สำเนาของ 02_explore.ipynb
copied: สำเนาของ 03_pipeline.ipynb
created README.md
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   01_setup_and_data.ipynb
	new file:   02_explore.ipynb
	new file:   03_pipeline.ipynb
	new file:   04_features.ipynb
	new file:   11111.ipynb
	new file:   344-341 Example-ReadAndWriteFile.ipynb
	new file:   README.md
	new file:   Untitled0.ipynb
	new file:   Untitled1.ipynb
	new file:   Untitled2.ipynb
	new file:   Untitl

In [35]:
import os
os.chdir('/content/exercise-pose')

# สร้าง .gitignore
with open('.gitignore', 'w') as f:
    f.write("""
# ไม่เอาไฟล์พวกนี้
Untitled*.ipynb
11111.ipynb
344-341 Example-ReadAndWriteFile.ipynb
*.pyc
__pycache__/
.ipynb_checkpoints/
""")

# ลบไฟล์ที่ไม่ต้องการออกจาก git (ไม่ได้ลบไฟล์จริง แค่เอาออกจาก tracking)
!git rm --cached "Untitled0.ipynb"
!git rm --cached "Untitled1.ipynb"
!git rm --cached "Untitled2.ipynb"
!git rm --cached "Untitled3.ipynb"
!git rm --cached "Untitled4.ipynb"
!git rm --cached "Untitled5.ipynb"
!git rm --cached "Untitled6.ipynb"
!git rm --cached "11111.ipynb"
!git rm --cached "344-341 Example-ReadAndWriteFile.ipynb"
!git rm --cached "สำเนาของ 02_explore.ipynb"
!git rm --cached "สำเนาของ 03_pipeline.ipynb"
!git rm --cached "สำเนาของ 04_features.ipynb"

# commit และ push
!git add .gitignore
!git commit -m "remove unused notebooks, add gitignore"
!git push origin main

fatal: pathspec 'Untitled0.ipynb' did not match any files
fatal: pathspec 'Untitled1.ipynb' did not match any files
fatal: pathspec 'Untitled2.ipynb' did not match any files
fatal: pathspec 'Untitled3.ipynb' did not match any files
fatal: pathspec 'Untitled4.ipynb' did not match any files
fatal: pathspec 'Untitled5.ipynb' did not match any files
fatal: pathspec 'Untitled6.ipynb' did not match any files
fatal: pathspec '11111.ipynb' did not match any files
fatal: pathspec '344-341 Example-ReadAndWriteFile.ipynb' did not match any files
rm 'สำเนาของ 02_explore.ipynb'
rm 'สำเนาของ 03_pipeline.ipynb'
rm 'สำเนาของ 04_features.ipynb'
[main 10e315d] remove unused notebooks, add gitignore
 3 files changed, 3 deletions(-)
 delete mode 100644 "\340\270\252\340\270\263\340\271\200\340\270\231\340\270\262\340\270\202\340\270\255\340\270\207 02_explore.ipynb"
 delete mode 100644 "\340\270\252\340\270\263\340\271\200\340\270\231\340\270\262\340\270\202\340\270\255\340\270\207 03_pipeline.ipynb"
 del

In [36]:
import os
os.chdir('/content/exercise-pose')

# เช็คไฟล์ที่อยู่ใน repo ตอนนี้
!git ls-files

.gitignore
01_setup_and_data.ipynb
02_explore.ipynb
03_pipeline.ipynb
04_features.ipynb
README.md


In [39]:
#ทำงานเสร็จแล้วอยาก push ขึ้น GitHub รัน cell นี้เลย
import os
os.chdir('/content/exercise-pose')

# 1. copy notebook ที่อัพเดทล่าสุดจาก Drive เข้า repo
import shutil
NOTEBOOK_DIR = '/content/drive/MyDrive/Colab Notebooks'

for nb in ['01_setup_and_data.ipynb','02_explore.ipynb',
           '03_pipeline.ipynb','04_features.ipynb']:
    src = f"{NOTEBOOK_DIR}/{nb}"
    dst = f"/content/exercise-pose/{nb}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"copied: {nb}")

# 2. push ขึ้น GitHub
!git add .
!git commit -m "update: ใส่คำอธิบายสั้นๆ ว่าทำอะไรไป"
!git push origin main

copied: 01_setup_and_data.ipynb
copied: 02_explore.ipynb
copied: 03_pipeline.ipynb
copied: 04_features.ipynb
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
